# Stage 16B: test-like branch-overlap validation

短い既知prefix、全suffix採点、target-free donor graph、通常/空間/branch-group foldを固定します。CPU-onlyで、学習・提出は行いません。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess
REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'
data_dir = drive_root / 'data'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
assert (data_dir / 'train').is_dir()
artifact_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)

## 8坑井smoke

最初にデータ契約、donor graph、全suffix control、hidden-target invarianceを小さく確認します。

In [ ]:
SMOKE_ID = 'stage16b_testlike_smoke_v003'
smoke_dir = artifact_dir / SMOKE_ID
if not (smoke_dir / 'summary.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-testlike-validation',
        '--config', 'configs/experiment/stage16b_testlike_validation.yaml',
        '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir),
        '--run-id', SMOKE_ID, '--limit-wells', '8',
    ], cwd=repo_dir, check=True)
smoke = json.loads((smoke_dir / 'summary.json').read_text())
assert smoke['stage16b_complete'] and smoke['hidden_target_invariance']
smoke

## 773坑井full validation manifest

既存runがあれば再利用します。GPUは不要です。

In [ ]:
RUN_ID = 'stage16b_testlike_validation_full_v003'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'summary.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-testlike-validation',
        '--config', 'configs/experiment/stage16b_testlike_validation.yaml',
        '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir), '--run-id', RUN_ID,
    ], cwd=repo_dir, check=True)
else:
    print('Reusing completed run:', run_dir)
summary = json.loads((run_dir / 'summary.json').read_text())
{
    'stage16b_complete': summary['stage16b_complete'],
    'n_wells': summary['n_wells'], 'n_cuts': summary['n_cuts'],
    'n_primary_cuts': summary['n_primary_cuts'],
    'n_donor_edges': summary['n_donor_edges'],
    'n_branch_groups': summary['n_branch_groups'],
    'hidden_target_invariance': summary['hidden_target_invariance'],
    'manifest_sha256': summary['manifest_sha256'],
    'fold_sha256': summary['fold_sha256'],
    'donor_structure_sha256': summary['donor_structure_sha256'],
    'branch_group_sha256': summary['branch_group_sha256'],
    'control_rmse': {name: value['pooled_rmse'] for name, value in summary['control_metrics'].items()},
    'next_step': summary['next_step'],
}

結果を共有してください。Stage 17ではこのmanifestを変更せず、凍結したV599 strong-baseを同じ疑似testへreplayします。